In [1]:
import heapq
import random

class Lot:
    def __init__(self, id, product_type, arrival_time):
        self.id = id
        self.product_type = product_type    # 1 or 2
        self.arrival_time = arrival_time
        self.current_step = 0
        self.status = "Waiting"
        self.assigned_tool = -1

In [2]:
class Step:
    def __init__(self, id, stage_time, run_time, workstation_id):
        self.id = id
        self.stage_time = stage_time
        self.run_time = run_time
        self.workstation_id = workstation_id

In [3]:
class Workstation:
    def __init__(self, id, num_tools):
        self.id = id
        self.num_tools = num_tools
        self.tool_free_until = [0.0] * num_tools

    def get_available_tool(self, current_time):
        for i, t in enumerate(self.tool_free_until):
            if t <= current_time:
                return i
        return -1


In [4]:

class Simulator:
    def __init__(self):
        self.current_time = 0.0
        self.sim_end_time = 1440.0  # 1 day
        self.events = []
        self.lots = []
        self.lot_counter = 0

        # Setup entities
        self.workstations = [Workstation(1,5), Workstation(2,3), Workstation(3,5)]
        self.steps = [
            Step(0, 1.0, 2.0, 1),    # S1: WS_A
            Step(1, 2.0, 1.0, 2),    # S2: WS_B
            Step(2, 1.5, 2.0, 2),    # S3: WS_B
            Step(3, 1.0, 1.5, 1),    # S4: WS_A
            Step(4, 2.0, 2.0, 3),    # S5: WS_C
            Step(5, 1.5, 1.0, 3)     # S6: WS_C
        ]
        self.product_types = {
            1: [0, 1, 2],        # Type1: S1,S2,S3
            2: [3, 4, 5]         # Type2: S4,S5,S6
        }
        # Schedule arrivals
        for i in range(130):
            pt = 1 if i < 80 else 2
            arrival_time = sum([random.expovariate(1.0) for _ in range(i+1)])
            heapq.heappush(self.events, (arrival_time, 'ARRIVE', pt))

    def get_gating_factor(self, lot):  # Customize for your logic
        return 0.5

    def should_gate(self, lot, step_id, gf):
        step = self.steps[step_id]
        ws = self.workstations[step.workstation_id-1]
        busy_tools = sum(1 for t in ws.tool_free_until if t > self.current_time)
        utilization = busy_tools / ws.num_tools
        return utilization > gf

    def find_available_tool(self, workstation_id):
        ws = self.workstations[workstation_id-1]
        return ws.get_available_tool(self.current_time)

    def run_simulation(self):
        while self.events and self.current_time < self.sim_end_time:
            self.current_time, event_type, data = heapq.heappop(self.events)
            if event_type == 'ARRIVE':
                self.on_arrive(data)
            elif event_type == 'GATING_CHECK':
                self.on_gating_check(data)
            elif event_type == 'PROCESS_START':
                self.on_process_start(data)
            elif event_type == 'PROCESS_COMPLETE':
                self.on_process_complete(data)
            elif event_type == 'EXIT':
                self.on_exit(data)

    def on_arrive(self, product_type):
        lot = Lot(self.lot_counter, product_type, self.current_time)
        self.lot_counter += 1
        self.lots.append(lot)
        heapq.heappush(self.events, (self.current_time, 'GATING_CHECK', lot))
        next_arrival = self.current_time + random.expovariate(1.0)
        heapq.heappush(self.events, (next_arrival, 'ARRIVE', product_type))

    def on_gating_check(self, lot):
        step_indices = self.product_types[lot.product_type]
        step_id = step_indices[lot.current_step]
        gf = self.get_gating_factor(lot)
        if self.should_gate(lot, step_id, gf):
            heapq.heappush(self.events, (self.current_time + 1, 'GATING_CHECK', lot))
        else:
            heapq.heappush(self.events, (self.current_time, 'PROCESS_START', lot))

    def on_process_start(self, lot):
        step_indices = self.product_types[lot.product_type]
        step_index = step_indices[lot.current_step]
        step = self.steps[step_index]
        ws = self.workstations[step.workstation_id-1]
        tool_index = self.find_available_tool(step.workstation_id)
        if tool_index == -1:
            heapq.heappush(self.events, (self.current_time + 1, 'PROCESS_START', lot))
            return
        lot.assigned_tool = tool_index
        process_time = step.stage_time + step.run_time
        ws.tool_free_until[tool_index] = self.current_time + process_time
        heapq.heappush(self.events, (self.current_time + process_time, 'PROCESS_COMPLETE', lot))

    def on_process_complete(self, lot):
        step_indices = self.product_types[lot.product_type]
        if lot.current_step < len(step_indices) - 1:
            lot.current_step += 1
            heapq.heappush(self.events, (self.current_time, 'GATING_CHECK', lot))
        else:
            heapq.heappush(self.events, (self.current_time, 'EXIT', lot))

    def on_exit(self, lot):
        pass  # You can collect statistics here




In [5]:
if __name__ == "__main__":
    sim = Simulator()
    sim.run_simulation()
    print("Simulation complete.")

Simulation complete.
